In [1]:
# Set Up

from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


# OPTION 1 - AML EXPLANATION ENGINE USING Local LLM via Ollama

In [2]:
# =============================================================================
# Ollama Installation (run once)
# This takes ~3-5 minutes on free Colab. You do NOT need to re-run it
# =============================================================================

!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 \
  --quiet

# Download the model weights directly from HuggingFace
!pip install huggingface_hub --quiet

from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF",
    filename="Llama-3.2-3B-Instruct-Q4_K_M.gguf",
    local_dir="/content/models"
)
print(f"Model downloaded to: {model_path}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.3/551.3 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Llama-3.2-3B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

Model downloaded to: /content/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf


In [ ]:
# =============================================================================
# OPTION 1 — AML EXPLANATION ENGINE (Local LLM via Ollama)
# -----------------------------------------------------------------------------
# Drop-in replacement for the Gemini version. Uses llama3.2:3b running locally
# via Ollama
#
# Prerequisites:
#   - Cell 1 (setup) must have been run in this session
# =============================================================================

import pandas as pd
import time
from pathlib import Path

# ------------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------------
OLLAMA_URL     = "http://localhost:11434/api/generate"
OLLAMA_MODEL   = "llama3.2:3b"

results_df = pd.read_csv('/content/gdrive/MyDrive/AML_Competition/models/model_output.csv')
isolation_df = pd.read_csv('/content/gdrive/MyDrive/AML_Competition/models/isolation_forest_results.csv')
# results_df[results_df['prediction'] == 1].shape[0]
# 61410 total customers
MAX_CUSTOMERS_PER_RUN = results_df.shape[0]
EXPLANATION_OUTPUT_PATH = Path("/content/gdrive/MyDrive/AML_Competition/models") / "model_output_explanations.csv"

# ------------------------------------------------------------------
# 2. LOAD AML INDICATOR LIBRARY
# ------------------------------------------------------------------
AML_LIBRARY_PATH = "/content/gdrive/MyDrive/AML_Competition/AML_RedFlags_Database.xlsx"

aml_library = pd.read_excel(AML_LIBRARY_PATH)
aml_library = aml_library.dropna(subset=["Red_Flag_Description"])
aml_library = aml_library[aml_library["Indicator_ID"].notna()].iloc[0:63]

# paths

BASE_DIR = '/content/gdrive/MyDrive/AML_Competition'
FEATURE_DIR = Path(BASE_DIR) / 'features'
df = pd.read_csv(FEATURE_DIR / 'customer_features_engineered.csv')

def build_library_summary(library_df):
    lines = []
    for _, row in library_df.iterrows():
        line = (
            f"[{row['Indicator_ID']}] {row['Indicator_Category']} > {row['Sub_Category']}: "
            f"{row['Red_Flag_Description']} "
            f"(Threshold: {row['Quantifiable_Threshold']}; "
            f"Tx Types: {row['Transaction_Type']}; "
            f"Risk Level: {row['Risk_Level']})"
        )
        lines.append(line)
    return "\n".join(lines)

LIBRARY_SUMMARY = build_library_summary(aml_library)

# ------------------------------------------------------------------
# 3. FEATURE CONTEXT BUILDER
# ------------------------------------------------------------------
FEATURE_LABELS = {
    "total_credit_amount":              "total money received",
    "total_debit_amount":               "total money sent",
    "avg_credit_amount":                "average incoming transaction size",
    "avg_debit_amount":                 "average outgoing transaction size",
    "max_credit_amount":                "largest single incoming transaction",
    "max_debit_amount":                 "largest single outgoing transaction",
    "credit_debit_ratio":               "ratio of money in vs money out",
    "transaction_count":                "total number of transactions",
    "credit_count":                     "number of incoming transactions",
    "debit_count":                      "number of outgoing transactions",
    "transactions_per_day":             "transactions per day",
    "unique_counterparties":            "number of different counterparties",
    "weekend_transaction_ratio":        "proportion of transactions on weekends",
    "night_transaction_ratio":          "proportion of transactions at night",
    "rapid_turnaround_count":           "number of rapid in-then-out fund movements",
    "avg_days_between_transactions":    "average days between transactions",
    "cash_transaction_ratio":           "proportion of cash transactions",
    "atm_transaction_ratio":            "proportion of ATM transactions",
    "international_transaction_ratio":  "proportion of international transactions",
    "wire_transaction_ratio":           "proportion of wire transfers",
    "structuring_indicator":            "structuring risk score",
    "round_amount_ratio":               "proportion of suspiciously round transaction amounts",
    "below_threshold_ratio":            "proportion of transactions just below reporting thresholds",
    "churn_ratio":                      "fund churn ratio (in vs out speed)",
    "income_to_credit_ratio":           "income vs actual credits ratio",
    "account_age_days":                 "how long the account has been open",
}

# Get all columns
all_cols = df.columns.tolist()

exclude_cols = [
    'customer_id',
    'label',
    'first_transaction_date',
    'last_transaction_date',
    'date',
    'transaction_datetime',
    'birth_date',
    'onboard_date',
    'established_date',
    'customer_type',
    'occupation',
    'industry',
    'day_name',
    'time_of_day',
    'year_month'
]

# Filter to only numeric columns
feature_cols = []
for col in all_cols:
    if col not in exclude_cols:
        # Check if column is numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            feature_cols.append(col)
        else:
            print(f"Skipping non-numeric column: {col} (type: {df[col].dtype})")


def extract_notable_features(customer_row, top_n=10):
    notes = []
    for feat in feature_cols:
        if feat not in customer_row.index:
            continue
        val = customer_row[feat]
        pop_mean = df[feat].mean()
        pop_std  = df[feat].std()
        if pop_std == 0:
            continue
        z = (val - pop_mean) / pop_std
        label = FEATURE_LABELS.get(feat, feat.replace("_", " "))
        notes.append((abs(z), z, label, val, pop_mean))

    notes.sort(key=lambda x: x[0], reverse=True)

    lines = []
    for _, z, label, val, avg in notes[:top_n]:
        direction = "higher" if z > 0 else "lower"
        lines.append(
            f"- {label.capitalize()}: {val:.2f} "
            f"({'%.2f' % abs(z)}x {direction} than the typical customer average of {avg:.2f})"
        )
    return "\n".join(lines) if lines else "No standout features identified."

# ------------------------------------------------------------------
# 4. PROMPT BUILDER  (unchanged from Gemini version)
# ------------------------------------------------------------------
SYSTEM_PROMPT = f"""You are an AML (Anti-Money Laundering) compliance expert writing risk
summaries for bank investigators. Your audience understands AML concepts but has NO
knowledge of machine learning or statistics.

You have access to the following AML Red Flag Indicator Library:
{LIBRARY_SUMMARY}

Your task: Given data about a customer and their model risk score, write a SHORT professional summary of no more than 3 short paragraphs (2000 characters):
1. States clearly whether the customer was flagged for AML review or not, and their risk level.
2. Explains, in plain terms, which specific behavioural patterns in their account data
   triggered concern — or why none did. Maps those patterns to the most relevant red flag indicators from the library above
   (cite the Indicator IDs in brackets, e.g. [CMLN-MULE-01]). Explains what these patterns could indicate in an AML context.
3. Ends with a clear recommended next step for the investigator.


Do NOT use headers, bullet points, or bold text in your response. Write in plain prose only.
Only mention terrorist financing if a terrorism-related indicator is explicitly matched.
Do NOT mention machine learning, isolation forest, anomaly scores, z-scores, or any
statistical jargon. Write as if you personally reviewed the account activity."""

def build_user_prompt(customer_id, risk_score, risk_category, prediction, feature_notes):
    flagged_text = "FLAGGED for AML review" if prediction == 1 else "NOT flagged for AML review"
    return f"""Customer ID: {customer_id}
Model Decision: {flagged_text}
Risk Level: {risk_category} (Risk Score: {risk_score:.3f})

Key account behaviours identified (compared to all customers):
{feature_notes}

Please write the AML risk explanation for this customer."""

# ------------------------------------------------------------------
# 5. LOCAL LLM CALL FUNCTION
# ------------------------------------------------------------------
from llama_cpp import Llama

# Load model once before the loop (not inside the function)
print("Loading model into memory...")
llm = Llama(
    model_path="/content/models/Llama-3.2-3B-Instruct-Q4_K_M.gguf",
    n_ctx=5012,
    n_gpu_layers=-1,   # offload all layers to GPU
    verbose=False
)
print("Model loaded.")

def call_local_llm(system_prompt, user_prompt, retries=3):
    try:
        response = llm.create_chat_completion(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ],
            max_tokens=512,
            temperature=0.3,
        )
        return response["choices"][0]["message"]["content"].strip()
    except Exception as e:
        return f"[ERROR: {str(e)}]"

# ------------------------------------------------------------------
# 6. MAIN EXPLANATION LOOP
# ------------------------------------------------------------------
print("=" * 70)
print("AML EXPLANATION ENGINE — Powered by Local LLM (llama3.2:3b)")
print("=" * 70)
print(f"\nAML Indicator Library loaded: {len(aml_library)} indicators")
print(f"Customers to explain: up to {MAX_CUSTOMERS_PER_RUN}")

results_to_explain = (
    results_df
    .sort_values(["predicted_label", "risk_score"], ascending=[False, False])
    .head(MAX_CUSTOMERS_PER_RUN)
    .copy()
)

print(f"Explaining {len(results_to_explain)} customers "
      f"({results_to_explain['predicted_label'].sum()} flagged, "
      f"{(results_to_explain['predicted_label']==0).sum()} not flagged)\n")

isolation_df = isolation_df.set_index("customer_id")
explanations = []
errors = []

for i, (_, row) in enumerate(results_to_explain.iterrows(), 1):
    customer_id   = row["customer_id"]
    risk_score    = row["risk_score"]
    prediction    = row["predicted_label"]

    customer_features = df[df["customer_id"] == customer_id]

    risk_category = isolation_df.loc[customer_id, "risk_category"] if customer_id in isolation_df.index else "Unknown"


    if customer_features.empty:
        feature_notes = "Detailed feature breakdown unavailable for this customer."
    else:
        feature_notes = extract_notable_features(customer_features.iloc[0])

    user_prompt = build_user_prompt(
        customer_id, risk_score, risk_category, prediction, feature_notes
    )

    print(f"[{i}/{len(results_to_explain)}] Customer {customer_id} | "
          f"{'FLAGGED' if prediction==1 else 'CLEAN':7s} | {risk_category} risk", end=" ... ")

    explanation_text = call_local_llm(SYSTEM_PROMPT, user_prompt)

    if explanation_text.startswith("[ERROR"):
        errors.append({"customer_id": customer_id, "error": explanation_text})
        print(f"FAILED — {explanation_text}")
    else:
        print("OK")

    explanations.append({
        "customer_id":   customer_id,
        "risk_score":    round(risk_score, 4),
        "risk_category": risk_category,
        "flagged":       bool(prediction),
        "explanation":   explanation_text
    })

    # Save after every customer so progress survives a disconnect
    pd.DataFrame(explanations).to_csv(EXPLANATION_OUTPUT_PATH, index=False)

# ------------------------------------------------------------------
# 7. SAVE RESULTS
# ------------------------------------------------------------------
explanations_df = pd.DataFrame(explanations)

print(f"\n{'='*70}")
print(f"EXPLANATIONS COMPLETE")
print(f"{'='*70}")
print(f"  Total explained : {len(explanations_df)}")
if not explanations_df.empty:
    print(f"  Flagged         : {explanations_df['flagged'].sum()}")
    print(f"  Not flagged     : {(~explanations_df['flagged']).sum()}")
else:
    print(f"  (No explanations generated — check errors above)")
print(f"  Errors          : {len(errors)}")
print(f"  Saved to        : {EXPLANATION_OUTPUT_PATH}")

if errors:
    print(f"\n  Failed customers: {[e['customer_id'] for e in errors]}")

# ------------------------------------------------------------------
# 8. PREVIEW — Top 3 Explanations
# ------------------------------------------------------------------
print(f"\n{'='*70}")
print("SAMPLE EXPLANATIONS (Top 3 by Risk Score)")
print(f"{'='*70}")

for _, row in explanations_df.head(3).iterrows():
    print(f"\nCustomer: {row['customer_id']}  |  Risk: {row['risk_category']}  "
          f"|  Flagged: {row['flagged']}")
    print("-" * 60)
    print(row["explanation"])
    print()

/tmp/ipython-input-1383290898.py:41: DtypeWarning: Columns (83,84,85,87,89) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FEATURE_DIR / 'customer_features_engineered.csv')


Skipping non-numeric column: occupation_code (type: object)
Skipping non-numeric column: industry_code (type: object)
Loading model into memory...


llama_context: n_ctx_per_seq (5012) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Model loaded.
AML EXPLANATION ENGINE — Powered by Local LLM (llama3.2:3b)

AML Indicator Library loaded: 63 indicators
Customers to explain: up to 61410
Explaining 61410 customers (615 flagged, 60795 not flagged)

[1/61410] Customer SYNID0200999280 | FLAGGED | Very High risk ... OK
[2/61410] Customer SYNID0200100125 | FLAGGED | Very High risk ... OK
[3/61410] Customer SYNID0200681268 | FLAGGED | Very High risk ... OK
[4/61410] Customer SYNID0200931678 | FLAGGED | Very High risk ... OK
[5/61410] Customer SYNID0200159270 | FLAGGED | Very High risk ... OK
[6/61410] Customer SYNID0200485667 | FLAGGED | Very High risk ... OK
[7/61410] Customer SYNID0200161910 | FLAGGED | Very High risk ... OK
[8/61410] Customer SYNID0200599266 | FLAGGED | Very High risk ... OK
[9/61410] Customer SYNID0200151421 | FLAGGED | Very High risk ... OK
[10/61410] Customer SYNID0200684443 | FLAGGED | Very High risk ... OK
[11/61410] Customer SYNID0200242911 | FLAGGED | Very High risk ... OK
[12/61410] Customer SYNID

# OPTION 2 - AML EXPLANATION ENGINE USING GEMINI 2.5 FLASH




In [ ]:
# =============================================================================
# OPTION 2 - AML EXPLANATION ENGINE USING GEMINI 2.5 FLASH
# -----------------------------------------------------------------------------
# This cell takes the Isolation Forest results and, for each customer, calls
# the Gemini API to generate a plain-language AML risk explanation grounded in
# your AML Red Flag Indicator Library. Explanations are written for AML
# analysts with no machine learning background.
#
# Prerequisites (run cells above first):
#   - results_df      : DataFrame with columns customer_id, risk_score,
#                       risk_category, prediction, anomaly_score
#   - df              : Original customer features DataFrame
#   - feature_cols    : List of numeric feature column names
# =============================================================================


import google.generativeai as genai
import pandas as pd
import time
import json
import re
from pathlib import Path

# ------------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------------
from google.colab import userdata
GEMINI_API_KEY = userdata.get('DefaultKey')
genai.configure(api_key=GEMINI_API_KEY)
model_gemini = genai.GenerativeModel("gemini-2.5-flash")   # free-tier model

# Rate-limit settings (gemini-2.0-flash free tier: 15 RPM / 20 RPD)
SECONDS_BETWEEN_CALLS = 5           # pause between each customer call
MAX_CUSTOMERS_PER_RUN = 1           # safety cap; increase if quota allows

# Output file
EXPLANATION_OUTPUT_PATH = Path("/content/gdrive/MyDrive/AML_Competition/models") / "aml_explanations_gemini.csv"

# ------------------------------------------------------------------
# 2. LOAD AML INDICATOR LIBRARY
# ------------------------------------------------------------------
AML_LIBRARY_PATH = "/content/gdrive/MyDrive/AML_Competition/AML_RedFlags_Database.xlsx"


aml_library = pd.read_excel(AML_LIBRARY_PATH)  # <-- change here
aml_library = aml_library.dropna(subset=["Red_Flag_Description"])
aml_library = aml_library[aml_library["Indicator_ID"].notna()]

# Build a compact, readable summary of the indicator library for the prompt
def build_library_summary(library_df):
    """Condense the AML library into a string suitable for a system prompt."""
    lines = []
    for _, row in library_df.iterrows():
        line = (
            f"[{row['Indicator_ID']}] {row['Indicator_Category']} > {row['Sub_Category']}: "
            f"{row['Red_Flag_Description']} "
            f"(Threshold: {row['Quantifiable_Threshold']}; "
            f"Tx Types: {row['Transaction_Type']}; "
            f"Risk Level: {row['Risk_Level']})"
        )
        lines.append(line)
    return "\n".join(lines)

LIBRARY_SUMMARY = build_library_summary(aml_library)

# paths

BASE_DIR = '/content/gdrive/MyDrive/AML_Competition'
FEATURE_DIR = Path(BASE_DIR) / 'features'
df = pd.read_csv(FEATURE_DIR / 'customer_features_engineered.csv')
results_df = pd.read_csv('/content/gdrive/MyDrive/AML_Competition/models/isolation_forest_results.csv')


# ------------------------------------------------------------------
# 3. FEATURE CONTEXT BUILDER
# ------------------------------------------------------------------
FEATURE_LABELS = {
    # Volume / amount
    "total_credit_amount":              "total money received",
    "total_debit_amount":               "total money sent",
    "avg_credit_amount":                "average incoming transaction size",
    "avg_debit_amount":                 "average outgoing transaction size",
    "max_credit_amount":                "largest single incoming transaction",
    "max_debit_amount":                 "largest single outgoing transaction",
    "credit_debit_ratio":               "ratio of money in vs money out",

    # Velocity / frequency
    "transaction_count":                "total number of transactions",
    "credit_count":                     "number of incoming transactions",
    "debit_count":                      "number of outgoing transactions",
    "transactions_per_day":             "transactions per day",
    "unique_counterparties":            "number of different counterparties",

    # Time patterns
    "weekend_transaction_ratio":        "proportion of transactions on weekends",
    "night_transaction_ratio":          "proportion of transactions at night",
    "rapid_turnaround_count":           "number of rapid in-then-out fund movements",
    "avg_days_between_transactions":    "average days between transactions",

    # Cash / channel
    "cash_transaction_ratio":           "proportion of cash transactions",
    "atm_transaction_ratio":            "proportion of ATM transactions",
    "international_transaction_ratio":  "proportion of international transactions",
    "wire_transaction_ratio":           "proportion of wire transfers",

    # Structuring / layering
    "structuring_indicator":            "structuring risk score",
    "round_amount_ratio":               "proportion of suspiciously round transaction amounts",
    "below_threshold_ratio":            "proportion of transactions just below reporting thresholds",
    "churn_ratio":                      "fund churn ratio (in vs out speed)",

    # Profile
    "income_to_credit_ratio":           "income vs actual credits ratio",
    "account_age_days":                 "how long the account has been open",
}

# Get all columns
all_cols = df.columns.tolist()

exclude_cols = [
    'customer_id',
    'label',
    'first_transaction_date',
    'last_transaction_date',
    'date',
    'transaction_datetime',
    'birth_date',
    'onboard_date',
    'established_date',
    'customer_type',
    'occupation',
    'industry',
    'day_name',
    'time_of_day',
    'year_month'
]

# Filter to only numeric columns
feature_cols = []
for col in all_cols:
    if col not in exclude_cols:
        # Check if column is numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            feature_cols.append(col)
        else:
            print(f"Skipping non-numeric column: {col} (type: {df[col].dtype})")


def extract_notable_features(customer_row, top_n=10):
    """
    Return the top_n most extreme feature values for a customer by comparing
    the customer's deviation against the population, expressed in plain English.
    """
    notes = []
    for feat in feature_cols:
        if feat not in customer_row.index:
            continue
        val = customer_row[feat]
        pop_mean = df[feat].mean()
        pop_std  = df[feat].std()
        if pop_std == 0:
            continue
        z = (val - pop_mean) / pop_std
        label = FEATURE_LABELS.get(feat, feat.replace("_", " "))
        notes.append((abs(z), z, label, val, pop_mean))

    notes.sort(key=lambda x: x[0], reverse=True)

    lines = []
    for _, z, label, val, avg in notes[:top_n]:
        direction = "higher" if z > 0 else "lower"
        lines.append(
            f"- {label.capitalize()}: {val:.2f} "
            f"({'%.2f' % abs(z)}x {direction} than the typical customer average of {avg:.2f})"
        )
    return "\n".join(lines) if lines else "No standout features identified."

# ------------------------------------------------------------------
# 4. PROMPT BUILDER
# ------------------------------------------------------------------
SYSTEM_PROMPT = f"""You are an AML (Anti-Money Laundering) compliance expert writing risk
summaries for bank investigators. Your audience understands AML concepts but has NO
knowledge of machine learning or statistics.

You have access to the following AML Red Flag Indicator Library:
{LIBRARY_SUMMARY}

Your task: Given data about a customer and their model risk score, write a concise,
professional explanation (maximum 2000 characters) that:
1. States clearly whether the customer was flagged for AML review or not, and their risk level.
2. Explains, in plain terms, which specific behavioural patterns in their account data
   triggered concern — or why none did.
3. Maps those patterns to the most relevant red flag indicators from the library above
   (cite the Indicator IDs in brackets, e.g. [CMLN-MULE-01]).
4. Explains what these patterns could indicate in an AML context.
5. Ends with a clear recommended next step for the investigator.

Do NOT mention machine learning, isolation forest, anomaly scores, z-scores, or any
statistical jargon. Write as if you personally reviewed the account activity."""

def build_user_prompt(customer_id, risk_score, risk_category, prediction, feature_notes):
    flagged_text = "FLAGGED for AML review" if prediction == 1 else "NOT flagged for AML review"
    return f"""Customer ID: {customer_id}
Model Decision: {flagged_text}
Risk Level: {risk_category} (Risk Score: {risk_score:.3f})

Key account behaviours identified (compared to all customers):
{feature_notes}

Please write the AML risk explanation for this customer."""

# ------------------------------------------------------------------
# 5. MAIN EXPLANATION LOOP
# ------------------------------------------------------------------
print("=" * 70)
print("AML EXPLANATION ENGINE — Powered by Gemini")
print("=" * 70)
print(f"\nAML Indicator Library loaded: {len(aml_library)} indicators")
print(f"Customers to explain: up to {MAX_CUSTOMERS_PER_RUN}")
print(f"Rate limit pause: {SECONDS_BETWEEN_CALLS}s between calls\n")

# Select customers to explain:
#   - Prioritise flagged (prediction=1) first, then by risk score descending
results_to_explain = (
    results_df
    .sort_values(["prediction", "risk_score"], ascending=[False, False])
    .head(MAX_CUSTOMERS_PER_RUN)
    .copy()
)

print(f"Explaining {len(results_to_explain)} customers "
      f"({results_to_explain['prediction'].sum()} flagged, "
      f"{(results_to_explain['prediction']==0).sum()} not flagged)\n")

explanations = []
errors       = []

for i, (_, row) in enumerate(results_to_explain.iterrows(), 1):
    customer_id   = row["customer_id"]
    risk_score    = row["risk_score"]
    risk_category = row["risk_category"]
    prediction    = row["prediction"]

    # Get full feature row for this customer
    customer_features = df[df["customer_id"] == customer_id]
    if customer_features.empty:
        feature_notes = "Detailed feature breakdown unavailable for this customer."
    else:
        feature_notes = extract_notable_features(customer_features.iloc[0])

    user_prompt = build_user_prompt(
        customer_id, risk_score, risk_category, prediction, feature_notes
    )

    print(f"[{i}/{len(results_to_explain)}] Customer {customer_id} | "
          f"{'FLAGGED' if prediction==1 else 'CLEAN':7s} | {risk_category} risk", end=" ... ")

    explanation_text = f"[ERROR: no response generated]"   # default if all attempts fail

    for attempt in range(3):   # try up to 3 times per customer
        try:
            response = model_gemini.generate_content(
                contents=[
                    {"role": "user", "parts": [SYSTEM_PROMPT + "\n\n" + user_prompt]}
                ],
                generation_config=genai.types.GenerationConfig(
                    temperature=0.3
                )
            )
            explanation_text = response.text.strip()
            print("OK")
            break   # success — exit the retry loop

        except Exception as e:
            err_str = str(e)
            if "429" in err_str:
                match = re.search(r"retry in ([\d.]+)s", err_str)
                wait = float(match.group(1)) + 2 if match else 60
                if attempt == 2:   # final attempt failed — record and move on
                    explanation_text = f"[ERROR: rate limit after 3 attempts]"
                    errors.append({"customer_id": customer_id, "error": err_str})
                    print("FAILED — gave up after 3 attempts")
                    break
                print(f"RATE LIMITED — waiting {wait:.0f}s then retrying (attempt {attempt+1}/3)...")
                time.sleep(wait)
            else:
                explanation_text = f"[ERROR: {err_str}]"
                errors.append({"customer_id": customer_id, "error": err_str})
                print(f"FAILED — {e}")
                break   # non-rate-limit error, don't retry

    # Append result for this customer (outside retry loop, always runs)
    explanations.append({
        "customer_id":   customer_id,
        "risk_score":    round(risk_score, 4),
        "risk_category": risk_category,
        "flagged":       bool(prediction),
        "explanation":   explanation_text
    })

    # Rate-limit pause (skip after last call)
    if i < len(results_to_explain):
        time.sleep(SECONDS_BETWEEN_CALLS)

# ------------------------------------------------------------------
# 6. SAVE RESULTS
# ------------------------------------------------------------------
explanations_df = pd.DataFrame(explanations)
explanations_df.to_csv(EXPLANATION_OUTPUT_PATH, index=False)

print(f"\n{'='*70}")
print(f"EXPLANATIONS COMPLETE")
print(f"{'='*70}")
print(f"  Total explained : {len(explanations_df)}")
if not explanations_df.empty:
    print(f"  Flagged         : {explanations_df['flagged'].sum()}")
    print(f"  Not flagged     : {(~explanations_df['flagged']).sum()}")
else:
    print(f"  (No explanations generated — check errors above)")
print(f"  Errors          : {len(errors)}")
print(f"  Saved to        : {EXPLANATION_OUTPUT_PATH}")

if errors:
    print(f"\n  Failed customers: {[e['customer_id'] for e in errors]}")

# ------------------------------------------------------------------
# 7. PREVIEW — Top 3 Explanations
# ------------------------------------------------------------------
print(f"\n{'='*70}")
print("SAMPLE EXPLANATIONS (Top 3 by Risk Score)")
print(f"{'='*70}")

for _, row in explanations_df.head(3).iterrows():
    print(f"\nCustomer: {row['customer_id']}  |  Risk: {row['risk_category']}  "
          f"|  Flagged: {row['flagged']}")
    print("-" * 60)
    print(row["explanation"])
    print()

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipython-input-4013482750.py:70: DtypeWarning: Columns (83,84,85,87,89) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FEATURE_DIR / 'customer_features_engineered.csv')


Skipping non-numeric column: occupation_code (type: object)
Skipping non-numeric column: industry_code (type: object)
AML EXPLANATION ENGINE — Powered by Gemini

AML Indicator Library loaded: 144 indicators
Customers to explain: up to 1
Rate limit pause: 5s between calls

Explaining 1 customers (1 flagged, 0 not flagged)

[1/1] Customer SYNID0200931678 | FLAGGED | Very High risk ... OK

EXPLANATIONS COMPLETE
  Total explained : 1
  Flagged         : 1
  Not flagged     : 0
  Errors          : 0
  Saved to        : /content/gdrive/MyDrive/AML_Competition/models/aml_explanations_gemini.csv

SAMPLE EXPLANATIONS (Top 3 by Risk Score)

Customer: SYNID0200931678  |  Risk: Very High  |  Flagged: True
------------------------------------------------------------
Customer SYNID0200931678 has been FLAGGED for AML review at a **Very High** risk level.

Our analysis of this account reveals highly unusual and concerning transactional patterns. The client exhibits an exceptionally high volume and val